# nanolab — train Qwen3-0.6B on gsm8k (GRPO + LoRA)

**Before running:** menu → *Runtime → Change runtime type → T4 GPU* → Save.

Then run the cells top to bottom. If Colab disconnects mid-training, reconnect,
re-run cells 1–3, and use the **resume** cell instead of the train cell.

In [ ]:
# 1) Get the code (private repo → paste a GitHub token when asked).
#    Create one at github.com/settings/tokens → 'Generate new token (classic)'
#    → tick the 'repo' scope → generate → copy.
from getpass import getpass
token = getpass('GitHub token: ')
!git clone https://{token}@github.com/khwahish1509/RLPost.git nanolab
%cd nanolab

In [ ]:
# 2) Install: uv + project deps + the GPU training stack (takes a few minutes)
import os
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv sync -q
!uv pip install -q torch transformers peft
!uv tool install prime

In [ ]:
# 3) Install the task environment + quick sanity check
!uv run nanolab env install primeintellect/gsm8k
!uv run pytest -q  # 29 tests incl. a GRPO direction check (torch is present here)

In [ ]:
# 4) TRAIN — 50 steps. Watch for:
#    'pre-flight baseline reward: 0.xxx'  (must be in the 10–80% window)
#    'step N reward 0.xxx loss y.yyyy'    (one line per step)
#    checkpoints land in adapters/ every 10 steps
!uv run nanolab train configs/qwen3-0.6b-gsm8k.toml

In [ ]:
# 4b) ONLY after a disconnect: continue where it stopped
#     (re-run cells 1–3 first; same batches get redrawn, nothing is lost)
!uv run nanolab train configs/qwen3-0.6b-gsm8k.toml --resume

In [ ]:
# 5) Bring the results home: downloads a zip with the trained adapter
#    checkpoints + the lab database. Unzip it into your local nanolab folder.
!zip -qr nanolab-artifacts.zip adapters results/nanolab.db
from google.colab import files
files.download('nanolab-artifacts.zip')